In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import sys
sys.path.append("..")

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


from src.model import (
    DHP,
    CHP, CHP_plot, CHP_IC, CHP_IC_plot,
    neg_log_likelihood, hawkes_residuals,
    neg_log_likelihood_bivariate, hawkes_residuals_bivariate,
    spectral_det, trace_1, trace_2,
    engle_russell_ed_test, _default_bivariate_bounds,
    fit_univariate, univariate_goodness_of_fit, pass_rate_univariate,
    simulate_bivariate, fit_bivariate, bivariate_goodness_of_fit,
    pass_rate_bivariate_sim,fit_bivariate_sum_exp,neg_log_likelihood_bivariate_sum_exp,
    hawkes_residuals_bivariate_sum_exp,bivariate_goodness_of_fit_sum_exp,_eval_window_sum_exp,
    qq_overlay,_sample_random_windows,_aggregate_pass_rates, _print_pass_rate_summary, pass_rate_by_window_size_bivariate, pass_rate_by_window_size_sum_exp,
    compare_kernels_by_window_size
)

from src.data import (
    build_hawkes_dataframe,
    extract_time_window,
    initialize_hawkes_params,
    initialize_hawkes_params_sum_exp,
)

In [27]:
from scipy.optimize import brentq
from scipy.optimize import minimize, NonlinearConstraint,LinearConstraint
from scipy.stats import norm
from scipy import stats


from scipy.stats import kstest, expon
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from tick.hawkes import SimuHawkesExpKernels

In [28]:
import nest_asyncio
from tardis_dev import download_datasets_async
import glob, pandas as pd

nest_asyncio.apply()

await download_datasets_async(
    exchange="binance",
    data_types=["trades"],
    from_date="2026-09-01",
    to_date="2026-09-02",
    symbols=["BTCUSDT"],
    download_dir="../data",
)


files = glob.glob("../data/*.csv.gz")
print("Fichiers téléchargés :", files)

df = pd.read_csv(files[0])

Fichiers téléchargés : ['../data/binance_trades_2026-09-01_BTCUSDT.csv.gz']


In [29]:
# Make the DataFrame workable for Hawkes processes

df_total_hawkes = build_hawkes_dataframe(df) 

# Extract a window of 5 mins at 1:00pm

df_hawkes = extract_time_window(df_total_hawkes['time_stamp'], df_total_hawkes['side'], start=13*3600, duration=5*60)

# Initialization of Hawkes Parameters : empirical way using the window

theta_init, bounds, constraints = initialize_hawkes_params(df_hawkes['time_stamp'], df_hawkes['side'])

In [ ]:
# Fit of a bivariate Hawkes on the BTC/USDT using exp kernel

resultat = fit_bivariate(df_hawkes,
                  theta_init=theta_init,
                  method='trust-constr',
                  bounds=bounds,
                  constraints=None,
                  verbose=False)


theta_estimate=resultat.x

/opt/python/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:737: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)


In [ ]:
# Bivariate goodness of fit (for exp_kernel) : u1,u2 are the residuals we test

(u1, u2), results = bivariate_goodness_of_fit(theta_estimate, df_hawkes, verbose=True)

--- Goodness of fit ---
KS Test               -> Stat: 0.0844 | p-value: 0.0000

Ljung-Box             -> Stat: 572.7390 | p-value: 0.0000

Empirical variance of residuals: 1.2835
Engle-Russell ED Test -> Stat Z: 3.2676 | p-value: 0.0011

KS Test               -> Stat: 0.0588 | p-value: 0.0003

Ljung-Box             -> Stat: 198.4351 | p-value: 0.0000

Empirical variance of residuals: 1.3334
Engle-Russell ED Test -> Stat Z: 4.1946 | p-value: 0.0000



In [ ]:
# Initialization of Hawkes parameters for sum of exp kernel 
# We do not need bounds and constraints, because they will be automatically chosen by during the fit : we only use trust-constr.

theta_init_sum = initialize_hawkes_params_sum_exp(df_hawkes['time_stamp'], df_hawkes['side'])

In [ ]:
# Fit of a bivariate Hawkes on the BTC/USDT using sum exp kernel

resultat_sum  = fit_bivariate_sum_exp(df_hawkes,
                          theta_init_sum,
                          verbose=False)


theta_estimate_sum=resultat_sum.x

/opt/python/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:737: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)
/opt/python/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)


`xtol` termination condition is satisfied.
Number of iterations: 667, function evaluations: 10260, CG iterations: 4440, optimality: 5.94e-04, constraint violation: 0.00e+00, execution time: 1.6e+02 s.


In [ ]:
# Bivariate goodness of fit (for sum exp kernel) : u1,u2 are the residuals we test

(v1,v2), results = bivariate_goodness_of_fit_sum_exp(theta_estimate_sum, df_hawkes, verbose=True)

--- Goodness of fit ---
KS Test       -> Stat: 0.0384 | p-value: 0.0850
Ljung-Box     -> Stat: 316.1812 | p-value: 0.0000
empirical variance of résidus : 0.9478
Engle-Russell ED Test -> Stat Z: -0.6014 | p-value: 0.5475
KS Test       -> Stat: 0.0361 | p-value: 0.0720
Ljung-Box     -> Stat: 109.1794 | p-value: 0.0000
empirical variance of résidus : 1.1076
Engle-Russell ED Test -> Stat Z: 1.3531 | p-value: 0.1760


In [ ]:
# QQ-plot for Dimension Buys
qq_overlay(v1, u1, "Buys")

# QQ-plot for Dimension Sells
qq_overlay(v2, u2, "Sells")

NameError: name 'stats' is not defined

In [ ]:
comparison = compare_kernels_by_window_size(df_hawkes['time_stamp'], df_hawkes['side'],
    sizes_min=(5, 10, 20),
    n_per_size=10,
    seed=42,
)